In [ ]:
%cd /kaggle/working
import os

REPO_URL = "https://github.com/hassanimtiaz158/echolyx-MVP.git"
REPO_DIR = "/kaggle/working/echo"

if os.path.isdir(f"{REPO_DIR}/.git"):
    print("Repo already present -> pulling latest changes")
    %cd {REPO_DIR}
    !git fetch origin main
    !git reset --hard origin/main
else:
    if os.path.isdir(REPO_DIR):
        print("Found a non-git 'echo' directory (stale/leftover) -- removing before clone")
        !rm -rf {REPO_DIR}
    print("Cloning fresh")
    !git clone {REPO_URL} echo
    %cd {REPO_DIR}

!git log -1 --oneline

print("\nInstalling dependencies...")
!pip install -q -r requirements.txt

In [ ]:
import os
import re
from collections import Counter
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
RAW = Path("/kaggle/working/echo/data/raw")
CHK = Path("/kaggle/working/echo/checkpoints")
RAW.mkdir(parents=True, exist_ok=True)
CHK.mkdir(parents=True, exist_ok=True)


def _link(dest: Path, target) -> bool:
    """Symlink dest -> target, replacing a stale symlink but never a real file/dir."""
    if target is None or not Path(target).exists():
        print(f"  !! {dest.name}: NOT FOUND")
        return False
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_symlink():
        os.unlink(dest)
    elif dest.exists():
        print(f"  !! {dest} exists as a real file/dir -- leaving it")
        return False
    os.symlink(str(target), str(dest), target_is_directory=Path(target).is_dir())
    print(f"  {dest.name} -> {target}")
    return True


# Auto-discover data under /kaggle/input by CONTENT signature, not by dataset
# slug or folder name -- keeps working no matter how the datasets are named
# or reorganized.

# 1) MIMII DUE/DG archive roots: any directory containing attributes_00.csv.
#    DUE (dev_fan -> fan) ships only section_00; DG (fan -> fan) ships
#    section_00/01/02 -- verified against both source zips, so the section
#    count reliably tells the two archives apart regardless of folder names.
mimii_roots = sorted({p.parent for p in INPUT_ROOT.rglob("attributes_00.csv")})
due_root, dg_root = None, None
for root in mimii_roots:
    is_dg = any(root.rglob("*section_01*")) or any(root.rglob("*section_02*"))
    print(f"MIMII root found: {root}  -> {'DG (fan_dg)' if is_dg else 'DUE (fan)'}")
    if is_dg:
        dg_root = dg_root or root
    else:
        due_root = due_root or root

# 1b) OPTIONAL: original MIMII / DCASE2020 Task 2 fan data (normal_id_XX_/
#     anomaly_id_XX_ filenames, no attributes_00.csv). Only present if you've
#     attached the daisukelab/dc2020task2 dataset under Data > Add Input --
#     training works fine without it, this just adds more real fan anomalies
#     when available.
dcase_id_pat = re.compile(r"^(normal|anomaly)_id_\d+_")
dcase_fan_counts = Counter()
for p in INPUT_ROOT.rglob("*"):
    if p.is_dir() and p.name.lower() == "fan":
        n = sum(1 for f in p.rglob("*.wav") if dcase_id_pat.match(f.name))
        if n:
            dcase_fan_counts[p] = n
dcase_fan_root = max(dcase_fan_counts, key=dcase_fan_counts.get) if dcase_fan_counts else None
if dcase_fan_root:
    print(f"DCASE2020 fan root found: {dcase_fan_root}  ({dcase_fan_counts[dcase_fan_root]} id_XX wavs)")
else:
    print("DCASE2020 fan root not found (optional -- attach daisukelab/dc2020task2 to include it)")

# 1c) OPTIONAL: raw MIMII 2019 layout, e.g. fan/id_00/{normal,abnormal}/
#     00000000.wav (aditya2402/audio-anomaly-fan-data or similar). Neither
#     DUE/DG nor the DCASE2020-flattened layout ever has a folder literally
#     named "abnormal", so that name alone is an unambiguous signature.
mimii_raw_hits = [p for p in INPUT_ROOT.rglob("*") if p.is_dir() and p.name.lower() == "abnormal"]
mimii_raw_counts = Counter(p.parent.parent for p in mimii_raw_hits)
mimii_raw_root = max(mimii_raw_counts, key=mimii_raw_counts.get) if mimii_raw_counts else None
if mimii_raw_root:
    print(f"Raw-MIMII-layout fan root found: {mimii_raw_root}  ({mimii_raw_counts[mimii_raw_root]} id_XX/abnormal dirs)")
else:
    print("Raw-MIMII-layout fan root not found (optional -- attach aditya2402/audio-anomaly-fan-data to include it)")

# 2) Real-world broken-fan holdout set: a directory literally named broken_fans.
broken_hits = [p for p in INPUT_ROOT.rglob("*") if p.is_dir() and p.name == "broken_fans"]
broken_root = broken_hits[0] if broken_hits else None

# 3) Freesound normal clips (your SOUND/ folder): the directory holding the
#    most standalone .mp3 files -- content-based so it doesn't matter what
#    the folder is actually named.
mp3_counts = Counter(p.parent for p in INPUT_ROOT.rglob("*.mp3"))
mp3_counts.pop(broken_root, None)
freesound_root = max(mp3_counts, key=mp3_counts.get) if mp3_counts else None
if freesound_root:
    print(f"Freesound root found: {freesound_root}  ({mp3_counts[freesound_root]} mp3 files)")

# 4) PANNs backbone checkpoint (matches Cnn14_mAP0.431.pt or ...pth).
ckpt_hits = list(INPUT_ROOT.rglob("Cnn14*.pt*"))
ckpt_root = ckpt_hits[0] if ckpt_hits else None

ok = True
ok &= _link(RAW / "mimii" / "fan", due_root)
ok &= _link(RAW / "mimii" / "fan_dg", dg_root)
ok &= _link(RAW / "freesound", freesound_root)
ok &= _link(RAW / "broken_fans", broken_root)
ok &= _link(CHK / "Cnn14_mAP=0.431.pth", ckpt_root)
if dcase_fan_root is not None:
    _link(RAW / "mimii" / "dcase2020_fan", dcase_fan_root)  # optional, not part of `ok`
if mimii_raw_root is not None:
    _link(RAW / "mimii" / "mimii2019_fan", mimii_raw_root)  # optional, not part of `ok`
print("\nAll links OK:", ok)
if not ok:
    raise RuntimeError(
        "Missing data mount(s) above -- attach the dataset(s) with the "
        "missing content under Data > Add Input, then re-run this cell."
    )

%cd /kaggle/working/echo
!python -m src.data.collect --config configs/config.yaml
!python -m src.train --config configs/config.yaml
!python -m src.evaluate --config configs/config.yaml
!python -m src.anomaly --config configs/config.yaml

In [ ]:
%cd /kaggle/working/echo
import os

os.environ["GIT_TERMINAL_PROMPT"] = "0"
!git config user.email "kaggle@echolyx.ai"
!git config user.name "Kaggle Bot"
!git add -A -- checkpoints artifacts

status = !git status --porcelain
if status:
    print("Changed files:")
    !git status --short
    !git commit -m "chore: update training results from Kaggle run"

    # Token comes from a Kaggle Secret (Add-ons > Secrets, name it GITHUB_TOKEN
    # and enable it for this notebook) -- never hardcode a token in the cell.
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    remote = f"https://{token}@github.com/hassanimtiaz158/echolyx-MVP.git"
    !git remote set-url origin {remote}
    !git push origin main
    !git remote set-url origin https://github.com/hassanimtiaz158/echolyx-MVP.git
    print("Pushed to GitHub!")
else:
    print("No changes to push.")